# BEV Perception — TensorRT + Benchmark (Kaggle T4)

Mac-la mudiyaadha vishayangal ingu odum: TensorRT engine build + latency benchmark
+ visual comparison.

**Settings check pannu:** Accelerator = `GPU T4 x2`, Internet = `ON`

nuScenes dataset thevai illa — `calib_bundle.npz`-la 50 real sample already iruku.

## 1. Setup — code extract + freshness check

In [ ]:
import os, glob, shutil

print("=== /kaggle/input ===")
for root, dirs, files in os.walk('/kaggle/input'):
    if root.replace('/kaggle/input', '').count(os.sep) > 2:
        continue
    print(root, "->", (files[:6] + ['...'] if len(files) > 6 else files))

os.makedirs('/kaggle/working/bev', exist_ok=True)

# Kaggle zip-ai upload panra podhu THAANA extract pannidum,
# so rendu case-aiyum handle pannurom.
zips = glob.glob('/kaggle/input/**/bev_code.zip', recursive=True)
markers = glob.glob('/kaggle/input/**/models/simplebev.py', recursive=True)

if zips:
    print("\nzip kedaichadhu:", zips[0])
    os.system(f"unzip -oq {zips[0]} -d /kaggle/working/bev")
elif markers:
    src = os.path.dirname(os.path.dirname(markers[0]))
    print("\nextract aagi irukku:", src)
    shutil.copytree(src, '/kaggle/working/bev', dirs_exist_ok=True)
else:
    raise SystemExit("Code kaanom! Add Input la bev-code serthiyaa nu paaru.")

os.chdir('/kaggle/working/bev')
print("\ncwd:", os.getcwd())
print("files:", sorted(os.listdir('.'))[:12])

# Pazhaya code irundha ellam fail aagum - ippove nirthidalaam
if 'TRT_HAS_CALIBRATOR' in open('export/calibrate_int8.py').read():
    print("\n[OK] pudhu code - TensorRT 10+ support iruku")
else:
    raise SystemExit("\n[STOP] PAZHAYA CODE! Dataset-la pudhu bev_code.zip "
                     "upload panni New Version create pannu.")

In [ ]:
# best.pth + calib_bundle.npz-ai idathula kondu varurom
import glob, shutil, os

for pattern, dest in [('**/best.pth', 'runs/simplebev/best.pth'),
                      ('**/calib_bundle.npz', 'export/calib_bundle.npz')]:
    if os.path.exists(dest):
        print(f"already here: {dest}")
        continue
    hits = glob.glob(f'/kaggle/input/{pattern}', recursive=True)
    assert hits, f"{pattern} kaanom"
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    shutil.copy(hits[0], dest)
    print(f"copied {hits[0]} -> {dest}")

In [ ]:
# Packages install.
#
# onnxruntime-gpu-la version tricky: latest (1.23+) CUDA 13 kekkum,
# Kaggle-la CUDA 12.x thaan -> silent-a CPU-ku vizhundhudum.
# So CUDA 12 build-ai try pannurom. Ellaam fail aana CPU build
# poduvom - ONNX row slow aagum, aana TensorRT-ku pirachanai illa.
import subprocess, sys

def pip(*args):
    return subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args],
                          capture_output=True).returncode

# pyquaternion - training/evaluate.py (decode_predictions) ku thevai
# onnxconverter-common - TRT 11-la FP16 builder flag illa,
# so ONNX-ai fp16-a convert panna ithu thevai
pip("timm", "onnx", "pyquaternion", "onnxconverter-common")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y",
                "onnxruntime", "onnxruntime-gpu"], capture_output=True)

for v in ["1.20.2", "1.20.0", "1.19.2"]:
    if pip(f"onnxruntime-gpu=={v}") == 0:
        print("installed onnxruntime-gpu", v)
        break
else:
    pip("onnxruntime")
    print("GPU build kedaikala - CPU onnxruntime install panniten")

pip("tensorrt", "pycuda")

import torch, tensorrt as trt
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("TensorRT", trt.__version__)
print("calibrator API:", hasattr(trt, "IInt8EntropyCalibrator2"))

# BuilderFlag-la enna iruku? TRT version-ku version maarum -
# TRT 11-la FP16 flag remove aagirukku.
flags = [f for f in dir(trt.BuilderFlag) if not f.startswith("_")]
print("BuilderFlag options:", flags)

try:
    import onnxruntime as ort
    print("onnxruntime", ort.__version__, "OK")
except ImportError:
    print("onnxruntime INSTALL AAGALA - ONNX row skip aagum, "
          "TensorRT thodarum")

## 2. ONNX export + verify

PyTorch model -> rendu ONNX graph (camera_backbone + bev_decoder).
Export aana model correct-a nu PyTorch-oda compare panni check pannum.

In [ ]:
!python -m export.export_onnx

In [ ]:
# ORT nijamaa CUDA use pannudha nu check.
# Provider list-la irundha pothaadhu - session create panni paakkanum,
# CUDA provider load fail aana silent-a CPU-ku vizhundhudum.
try:
    import onnxruntime as ort
    s = ort.InferenceSession("export/onnx/camera_backbone.onnx",
                             providers=["CUDAExecutionProvider",
                                        "CPUExecutionProvider"])
    print("ORT actually using:", s.get_providers()[0])
except ImportError:
    print("onnxruntime illa - skip. TensorRT-ku ithu thevai illa.")

## 3. TensorRT engines

FP32 -> FP16 -> INT8. Ovvondrum ~5 min edukkum.

**TensorRT 11-la precision epdi set pannurom?**
Pazhaya TRT-la `config.set_flag(BuilderFlag.FP16)` sonna pothum.
TRT 11-la antha flag-e remove pannitaanga - ippo "strongly typed",
ONNX-la enna type irukko adhe TRT use pannum. So:

- **FP16**: ONNX-ai `onnxconverter_common`-la fp16-a convert pannurom
- **INT8**: ONNX-la QDQ nodes potrom (explicit quantization)

**INT8 fail aagalaam - paravaayilla.** Mac-la test panna podhu INT8 PTQ
intha model-oda detections-ai keduthudhu (agreement 0-68%). Karanam:
EfficientNet depthwise conv + model weak-a train aagirukku (NDS 0.043).
Odi paathu, result-ai honest-a report pannuvom.

In [ ]:
!python -m export.calibrate_int8 --precision fp32

In [ ]:
!python -m export.calibrate_int8 --precision fp16

In [ ]:
!python -m export.calibrate_int8 --precision int8 || echo 'INT8 FAILED - paravaayilla, FP16 use pannuvom'

## 4. Benchmark — full table

In [ ]:
!python -m export.benchmark

## 5. Visual comparison

Numbers table nallathu, aana "quantization apram boxes appadiye irukka?"
nu neradiya kaatta padam venum.

**Box agreement** = FP32 sonna box-ku quantized model-la 0.5m ulla
same-class box irukka? 95%+ vandha pathiram.

In [ ]:
!python -m visualization.compare_renderer -a pytorch -b trt_fp16 --frames 6 --gif

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('runs/compare/compare_*.png'))[:3]:
    display(Image(filename=p))

In [ ]:
# INT8-um compare pannuvom - engine build aagirundha mattum.
# os.system use pannurom ('!' if-block ulla confusion vendaam)
import os
if os.path.exists('export/engines/camera_backbone_int8.plan'):
    os.system('python -m visualization.compare_renderer -a pytorch '
              '-b trt_int8 --frames 4 --out-dir runs/compare_int8')
else:
    print("INT8 engine illa - skip")

## 6. Ellaa result-um ore edathula

In [ ]:
import os

print("=== LATENCY TABLE ===")
print(open('runs/benchmark.md').read())

for label, path in [("FP16 vs FP32", 'runs/compare/summary.md'),
                    ("INT8 vs FP32", 'runs/compare_int8/summary.md')]:
    if os.path.exists(path):
        print(f"\n=== {label} ===")
        print(open(path).read())

print("\n=== ENGINE SIZES ===")
if os.path.exists('export/engines'):
    for f in sorted(os.listdir('export/engines')):
        if f.endswith('.plan'):
            print(f"{f:42s} {os.path.getsize('export/engines/' + f) / 1e6:7.1f} MB")

## 8. NVIDIA Triton — model-ai SERVICE-a deploy pannurathu

Ippo varaikum model namma script ulla thaan odunuchu. Production-la
appdi illa — model oru **server**-la irukkum, vera applications
network vazhiya request anuppum.

**Triton** = NVIDIA-oda production inference server. Batching, multiple
model instances, metrics ellam built-in.

**Yaen PyTriton, saadha Triton illa?**
Saadha Triton Docker container-la varum (~15 GB image) — Kaggle-la
Docker illa. PyTriton = **ATHE Triton server**, aana Python process
ulla odum. Server nijam, HTTP/gRPC endpoint nijam, dynamic batching
nijam — Docker mattum thaan illa.

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "nvidia-pytriton"], capture_output=True)
print("install rc:", r.returncode)
if r.returncode:
    print(r.stderr.decode()[-800:])

import pytriton
print("pytriton", pytriton.__version__ if hasattr(pytriton, "__version__") else "OK")

### Server start + live demo

Intha cell:
1. Triton server-ai start pannum (TensorRT engines vachi)
2. Ovvoru frame-aiyum **client vazhiya server-ku** anuppum
3. Nadakkuradhai padama varaiyum → GIF

`--instances 2` = model-oda 2 parallel copy. Real Triton-la ithu
`config.pbtxt`-la `instance_group { count: 2 }`.

In [ ]:
!python -m triton_deploy.triton_demo --frames 16 --precision int8 --instances 2

In [ ]:
from IPython.display import Image, display
import glob, os
for p in sorted(glob.glob('runs/triton_demo/frame_*.png'))[:2]:
    display(Image(filename=p))
if os.path.exists('runs/triton_demo/summary.md'):
    print(open('runs/triton_demo/summary.md').read())

### INT8 engine illaina FP32 vachi odu

INT8 build aagalaina intha cell odu (illaina skip pannu).

In [ ]:
import os
if not os.path.exists('runs/triton_demo/demo.gif'):
    os.system('python -m triton_deploy.triton_demo --frames 16 '
              '--precision fp32 --instances 2')
else:
    print("INT8 demo already aachu - skip")

## 9. Download

Right panel -> **Output** -> intha file-gal download pannu:

**Benchmark**
- `runs/benchmark.md` — latency table (PyTorch / ONNX / TRT FP32-FP16-INT8)
- `runs/compare/summary.md` — FP16 vs FP32
- `runs/compare_int8/summary.md` — INT8 vs FP32

**Padam / GIF**
- `runs/compare/compare.gif` — FP32 vs quantized, pakkam-pakkam
- `runs/triton_demo/demo.gif` — **Triton live demo** (ithu thaan mukkiyam)
- `runs/triton_demo/frame_*.png`
- `runs/triton_demo/summary.md` — Triton latency

**Engines** (HF Space deploy-ku thevaipadum)
- `export/engines/*.plan`
- `export/onnx/*.onnx`

Ithai enakku anuppu — README-la naan ellathaiyum serkiren.